In [1]:
import os
import random
import shutil
from glob import glob
from tqdm import tqdm
import albumentations as A
import cv2

/usr/local/lib/python3.11/dist-packages/albumentations/__init__.py:28: UserWarning: A new version of AlbumentationsX (2.0.18) is available! Your version is 2.0.9. Upgrade using: pip install -U albumentationsx
  check_for_updates()


In [2]:
def aplicar_aumentos_e_salvar(imagem_path, destino_dir, contador, transform, classe_nome):
    imagem = cv2.imread(imagem_path)
    nome_base = classe_nome  # Usar o nome da classe para o nome da imagem aumentada
    extensao = os.path.splitext(imagem_path)[1]

    nome_novo = f"{nome_base}_aug{contador}{extensao}"  # Nome único com contador
    caminho_destino = os.path.join(destino_dir, nome_novo)

    # Gerar e salvar a imagem aumentada
    imagem_aumentada = transform(image=imagem)['image']        
    cv2.imwrite(caminho_destino, imagem_aumentada)

    return contador + 1  # Retorna o próximo contador

def criar_dataset_com_aumento(dataset1_path, dataset3_path, percentual, seed):
    random.seed(seed)
    
    train_path = os.path.join(dataset1_path, "train")

    novo_dataset_path = dataset3_path

    # Copiar estrutura original (train/val/test e subpastas de classe)
    for subset in ["train", "val", "test"]:
        origem_subset = os.path.join(dataset1_path, subset)
        destino_subset = os.path.join(novo_dataset_path, subset)
        os.makedirs(destino_subset, exist_ok=True)

        for classe in os.listdir(origem_subset):
            origem_classe = os.path.join(origem_subset, classe)
            destino_classe = os.path.join(destino_subset, classe)
            os.makedirs(destino_classe, exist_ok=True)
            for imagem in glob(os.path.join(origem_classe, "*")):
                shutil.copy(imagem, destino_classe)
    aug = 2
    # Aumentação apenas no train
    if (aug == 2):
        transform = A.Compose([
            A.Resize(
                256,
                256,
                interpolation=cv2.INTER_CUBIC),
            A.HorizontalFlip(p=0.5), # AUG2
            A.RandomCrop(width=256, height=256, p=0.5), # AUG2
            A.Affine(scale=(0.5, 1.2), rotate=(-90, 90), p=0.5), # AUG2
            A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5), # AUG2
        ])
    else:
        transform = A.Compose([
            A.RandomBrightnessContrast(p=0.5), # AUG1
            A.HorizontalFlip(p=0.5), # AUG1
            A.ShiftScaleRotate(p=0.7), # AUG1
            A.RGBShift(p=0.5), # AUG1
            A.RandomGamma(p=0.5), # AUG1
        ])

    destino_train = os.path.join(novo_dataset_path, "train")

    # Calcular a quantidade de aumentações a serem feitas para todas as classes
    for classe in os.listdir(train_path):
        origem_classe = os.path.join(train_path, classe)
        destino_classe = os.path.join(destino_train, classe)
        os.makedirs(destino_classe, exist_ok=True)
        imagens = glob(os.path.join(origem_classe, "*"))
        qtd_original = len(imagens)

        # Calculando a quantidade de aumentos para **todas as classes** com base na porcentagem
        qtd_aumentar = int((percentual / 100) * 350)

        # Garantir que a quantidade total seja proporcional ao aumento
        qtd_total_imagens = qtd_original + qtd_aumentar

        print(f"Classe: {classe}, Imagens a aumentar: {qtd_aumentar} (Total: {qtd_total_imagens})")

        # Iniciar o contador
        contador = 0

        # Aumentar imagens
        for _ in range(qtd_aumentar):
            imagem_escolhida = random.choice(imagens)
            contador = aplicar_aumentos_e_salvar(imagem_escolhida, destino_classe, contador, transform, classe)

In [3]:
seeds = [1111, 1313, 1515, 1717, 1919]
percents = [1000]

for seed in seeds:
    print(f"Criando datasets para seed:    {seed}")
    for percent in percents:
        input_ds  = f'DS_REAL_SPLITED_{seed}_SYN_{percent}'
        output_ds = f'DS_REAL_SPLITED_{seed}_SYN_{percent}_DDPM_AUG2'
        criar_dataset_com_aumento(input_ds, output_ds, percent, seed)
        print(f'\n DS Criado {output_ds}\n')

Criando datasets para seed:    1111
Classe: BULKCARRIER, Imagens a aumentar: 3500 (Total: 7350)
Classe: CONTAINERSHIP, Imagens a aumentar: 3500 (Total: 7350)
Classe: GENERALCARGO, Imagens a aumentar: 3500 (Total: 7350)
Classe: OILPRODUCTSTANKER, Imagens a aumentar: 3500 (Total: 7350)
Classe: PASSENGERSSHIP, Imagens a aumentar: 3500 (Total: 7350)
Classe: TANKER, Imagens a aumentar: 3500 (Total: 7350)
Classe: TRAWLER, Imagens a aumentar: 3500 (Total: 7350)
Classe: TUG, Imagens a aumentar: 3500 (Total: 7350)
Classe: VEHICLESCARRIER, Imagens a aumentar: 3500 (Total: 7350)
Classe: YACHT, Imagens a aumentar: 3500 (Total: 7350)

 DS Criado DS_REAL_SPLITED_1111_SYN_1000_DDPM_AUG2

Criando datasets para seed:    1313
Classe: BULKCARRIER, Imagens a aumentar: 3500 (Total: 7349)
Classe: CONTAINERSHIP, Imagens a aumentar: 3500 (Total: 7350)
Classe: GENERALCARGO, Imagens a aumentar: 3500 (Total: 7350)
Classe: OILPRODUCTSTANKER, Imagens a aumentar: 3500 (Total: 7350)
Classe: PASSENGERSSHIP, Imagens a